# Multivariate Relationships: Interactions Between Heat Stress Metrics

This notebook analyzes relationships and interactions between different heat stress variables:

**Analysis Types:**
1. VPD vs Temperature scatter plots and correlations
2. Daytime vs Nighttime heat relationships
3. Joint probability distributions
4. Composite analysis (high VPD + high temp)
5. Principal Component Analysis (PCA)
6. Bivariate density plots
7. Conditional distributions
8. Multi-metric stress index development

**Goal:** Understand how multiple stressors combine to create compound heat stress

In [ ]:
# Standard imports
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import seaborn as sns
from pathlib import Path
from scipy import stats
from scipy.stats import gaussian_kde
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

print("Multivariate analysis imports complete!")

In [ ]:
# Configuration
DATA_DIR = Path('../..')  # Up two levels from notebooks/03_analysis/ to research/
NIGHTTIME_DIR = DATA_DIR / 'processed_nighttime_recovery'
DAYTIME_DIR = DATA_DIR / 'processed_daytime_heat'
VPD_DIR = DATA_DIR / 'processed_vpd'
MASK_FILE = DATA_DIR / 'masks/region_mask.nc'
OUTPUT_DIR = Path('../../figures/multivariate')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Focus states (Regions 4 & 6: major cattle-producing states)
# Note: IDs are from region_mask.nc (alphabetical order, not standard FIPS)
FOCUS_STATES = {
    'Alabama': 1,
    'Arizona': 3,
    'Florida': 8,
    'Georgia': 9,
    'Kentucky': 15,
    'Louisiana': 16,
    'Mississippi': 23,
    'New Mexico': 30,
    'North Carolina': 25,
    'Oklahoma': 34,
    'South Carolina': 38,
    'Tennessee': 40,
    'Texas': 41
}

print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

In [ ]:
# Helper functions
def load_monthly_data(data_dir, year_start=2010, year_end=2023):
    """Load monthly data for a given period."""
    files = sorted(data_dir.glob('*.nc'))
    datasets = []
    for f in files:
        year_month = f.stem.split('_')[-1]
        year = int(year_month[:4])
        if year_start <= year <= year_end:
            datasets.append(xr.open_dataset(f))
    if datasets:
        return xr.concat(datasets, dim='time')
    return None

def compute_state_mean(data, state_id, _state_mask=None):
    """Compute spatial mean for a specific state.
    
    Converts timedelta64[ns] to float hours if needed.
    
    Parameters
    ----------
    data : xarray.DataArray
        Data to compute mean for
    state_id : int
        State ID from region_mask
    _state_mask : ignored
        Deprecated parameter for backwards compatibility (state_mask is global)
    """
    mask = state_mask == state_id
    masked_data = data.where(mask)
    state_mean = masked_data.mean(dim=['lat', 'lon'])
    
    # Convert timedelta to hours if data is stored as timedelta
    if state_mean.dtype.kind == 'm':  # 'm' = timedelta
        # Convert nanoseconds to hours: divide by (3600 * 10^9)
        state_mean = state_mean / np.timedelta64(1, 'h')
        state_mean = state_mean.astype(np.float64)
    
    return state_mean

print("Helper functions defined!")

In [ ]:
# Load data
ds_mask = xr.open_dataset(MASK_FILE)
state_mask = ds_mask.state_mask
ds_mask.close()

print("Loading datasets (2010-2023)...")
ds_night = load_monthly_data(NIGHTTIME_DIR, 2010, 2023)
ds_day = load_monthly_data(DAYTIME_DIR, 2010, 2023)
ds_vpd = load_monthly_data(VPD_DIR, 2010, 2023)
print("Data loaded!")

## Plot 1: VPD vs Temperature Scatter Plot

Relationship between daytime heat and afternoon VPD.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

# Filter for summer months
summer_day = ds_day['hours_above_30'].where(ds_day.time.dt.month.isin([6, 7, 8]), drop=True)
summer_vpd = ds_vpd['vpd_mean'].where(ds_vpd.time.dt.month.isin([6, 7, 8]), drop=True)

for idx, (state_name, state_id) in enumerate(FOCUS_STATES.items()):
    ax = axes[idx]
    
    # Get state means
    temp_data = compute_state_mean(summer_day, state_id, state_mask)
    vpd_data = compute_state_mean(summer_vpd, state_id, state_mask)
    
    # Clean data
    valid = (~np.isnan(temp_data.values)) & (~np.isnan(vpd_data.values))
    temp_clean = temp_data.values[valid]
    vpd_clean = vpd_data.values[valid]
    
    # Scatter with density coloring
    xy = np.vstack([temp_clean, vpd_clean])
    z = gaussian_kde(xy)(xy)
    
    scatter = ax.scatter(temp_clean, vpd_clean, c=z, s=10, alpha=0.5, cmap='YlOrRd')
    
    # Add regression line
    z_poly = np.polyfit(temp_clean, vpd_clean, 2)
    p = np.poly1d(z_poly)
    x_range = np.linspace(temp_clean.min(), temp_clean.max(), 100)
    ax.plot(x_range, p(x_range), 'b--', linewidth=2, label='Quadratic fit')
    
    # Compute correlation
    corr, p_val = stats.pearsonr(temp_clean, vpd_clean)
    
    ax.set_xlabel('Daytime Hours > 30°C')
    ax.set_ylabel('Afternoon Mean VPD (kPa)')
    ax.set_title(f'{state_name}\nr = {corr:.3f} (p < 0.001)', fontweight='bold')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    
    plt.colorbar(scatter, ax=ax, label='Density')

plt.suptitle('Temperature-VPD Relationship (Summer 2010-2023)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '01_vpd_temperature_scatter.png', dpi=300, bbox_inches='tight')
plt.show()
print("Plot 1 saved!")

## Plot 2: Daytime vs Nighttime Heat Relationship

How daytime heat affects nighttime recovery.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

summer_day_heat = ds_day['hours_above_30'].where(ds_day.time.dt.month.isin([6, 7, 8]), drop=True)
summer_night_poor = ds_night['hours_above_24'].where(ds_night.time.dt.month.isin([6, 7, 8]), drop=True)

for idx, (state_name, state_id) in enumerate(FOCUS_STATES.items()):
    ax = axes[idx]
    
    day_data = compute_state_mean(summer_day_heat, state_id, state_mask)
    night_data = compute_state_mean(summer_night_poor, state_id, state_mask)
    
    # Clean data
    valid = (~np.isnan(day_data.values)) & (~np.isnan(night_data.values))
    day_clean = day_data.values[valid]
    night_clean = night_data.values[valid]
    
    # 2D histogram for density
    h = ax.hist2d(day_clean, night_clean, bins=30, cmap='YlOrRd', cmin=1)
    plt.colorbar(h[3], ax=ax, label='Number of Days')
    
    # Add regression line
    z = np.polyfit(day_clean, night_clean, 1)
    p = np.poly1d(z)
    x_range = np.linspace(day_clean.min(), day_clean.max(), 100)
    ax.plot(x_range, p(x_range), 'b--', linewidth=2, 
           label=f'y = {z[0]:.2f}x + {z[1]:.2f}')
    
    # Correlation
    corr, _ = stats.pearsonr(day_clean, night_clean)
    
    ax.set_xlabel('Daytime Hours > 30°C')
    ax.set_ylabel('Nighttime Hours > 24°C (Poor Recovery)')
    ax.set_title(f'{state_name}\nr = {corr:.3f}', fontweight='bold')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)

plt.suptitle('Daytime Heat vs Nighttime Recovery (Summer 2010-2023)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '02_day_night_relationship.png', dpi=300, bbox_inches='tight')
plt.show()
print("Plot 2 saved!")

## Plot 3: Joint Probability Distribution

Combined occurrence probabilities of multiple stressors.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

for idx, (state_name, state_id) in enumerate(FOCUS_STATES.items()):
    ax = axes[idx]
    
    temp_data = compute_state_mean(summer_day_heat, state_id, state_mask)
    vpd_data = compute_state_mean(summer_vpd, state_id, state_mask)
    
    # Clean data
    valid = (~np.isnan(temp_data.values)) & (~np.isnan(vpd_data.values))
    temp_clean = temp_data.values[valid]
    vpd_clean = vpd_data.values[valid]
    
    # Define bins
    temp_bins = [0, 4, 8, 12]  # hours > 30°C
    vpd_bins = [0, 2, 3, 5]  # kPa
    
    # Create joint probability matrix
    joint_prob = np.zeros((len(temp_bins)-1, len(vpd_bins)-1))
    
    for i in range(len(temp_bins)-1):
        for j in range(len(vpd_bins)-1):
            in_bin = ((temp_clean >= temp_bins[i]) & (temp_clean < temp_bins[i+1]) &
                     (vpd_clean >= vpd_bins[j]) & (vpd_clean < vpd_bins[j+1]))
            joint_prob[i, j] = np.sum(in_bin) / len(temp_clean) * 100
    
    # Plot heatmap
    im = ax.imshow(joint_prob.T, cmap='YlOrRd', aspect='auto', origin='lower',
                   vmin=0, vmax=joint_prob.max())
    
    # Labels
    ax.set_xticks(range(len(temp_bins)-1))
    ax.set_xticklabels([f'{temp_bins[i]}-{temp_bins[i+1]}' for i in range(len(temp_bins)-1)])
    ax.set_yticks(range(len(vpd_bins)-1))
    ax.set_yticklabels([f'{vpd_bins[i]}-{vpd_bins[i+1]}' for i in range(len(vpd_bins)-1)])
    ax.set_xlabel('Daytime Hours > 30°C')
    ax.set_ylabel('Afternoon VPD (kPa)')
    ax.set_title(f'Joint Probability: {state_name}', fontweight='bold')
    
    # Add percentage labels
    for i in range(len(temp_bins)-1):
        for j in range(len(vpd_bins)-1):
            text = ax.text(i, j, f'{joint_prob[i, j]:.1f}%',
                          ha='center', va='center', color='black', fontsize=10)
    
    plt.colorbar(im, ax=ax, label='Probability (%)')

plt.suptitle('Joint Probability Distribution: Temperature & VPD (Summer 2010-2023)',
            fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '03_joint_probability.png', dpi=300, bbox_inches='tight')
plt.show()
print("Plot 3 saved!")

## Plot 4: Composite Analysis (Extreme Conditions)

Compare conditions during compound vs single-stressor events.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

state_name = 'Texas'
state_id = FOCUS_STATES[state_name]

# Get all variables
temp_day = compute_state_mean(summer_day_heat, state_id, state_mask)
temp_night = compute_state_mean(summer_night_poor, state_id, state_mask)
vpd = compute_state_mean(summer_vpd, state_id, state_mask)

# Define extreme thresholds (95th percentile)
temp_day_p95 = np.nanpercentile(temp_day.values, 95)
vpd_p95 = np.nanpercentile(vpd.values, 95)

# Identify event types
compound_extreme = (temp_day.values > temp_day_p95) & (vpd.values > vpd_p95)
temp_only = (temp_day.values > temp_day_p95) & (vpd.values <= vpd_p95)
vpd_only = (temp_day.values <= temp_day_p95) & (vpd.values > vpd_p95)
neither = (temp_day.values <= temp_day_p95) & (vpd.values <= vpd_p95)

event_types = [
    ('Compound\n(High T + High VPD)', compound_extreme, 'darkred'),
    ('High Temp Only', temp_only, 'orange'),
    ('High VPD Only', vpd_only, 'blue'),
    ('Neither', neither, 'gray')
]

# Plot 1: Event frequency
ax = axes[0]
frequencies = [np.sum(mask) for _, mask, _ in event_types]
labels = [label for label, _, _ in event_types]
colors = [color for _, _, color in event_types]

bars = ax.bar(range(len(labels)), frequencies, color=colors, alpha=0.7, edgecolor='black')
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels)
ax.set_ylabel('Number of Days')
ax.set_title(f'Event Type Frequency: {state_name}', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, frequencies):
    ax.text(bar.get_x() + bar.get_width()/2., val,
           f'{val}\n({val/len(temp_day)*100:.1f}%)',
           ha='center', va='bottom', fontsize=9)

# Plot 2: Average nighttime recovery by event type
ax = axes[1]
night_by_event = []
for label, mask, color in event_types:
    if np.sum(mask) > 0:
        avg_night = np.nanmean(temp_night.values[mask])
        night_by_event.append(avg_night)
    else:
        night_by_event.append(0)

bars = ax.bar(range(len(labels)), night_by_event, color=colors, alpha=0.7, edgecolor='black')
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels)
ax.set_ylabel('Avg Nighttime Hours > 24°C')
ax.set_title('Poor Nighttime Recovery by Event Type', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, night_by_event):
    if val > 0:
        ax.text(bar.get_x() + bar.get_width()/2., val,
               f'{val:.1f}h', ha='center', va='bottom', fontsize=9)

# Plot 3: Distribution comparison
ax = axes[2]
for label, mask, color in event_types:
    if np.sum(mask) > 10:  # Only plot if enough data
        data = temp_day.values[mask]
        ax.hist(data, bins=20, alpha=0.5, label=label, color=color, edgecolor='black')

ax.set_xlabel('Daytime Hours > 30°C')
ax.set_ylabel('Frequency')
ax.set_title('Temperature Distribution by Event Type', fontweight='bold')
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

# Plot 4: VPD distribution comparison
ax = axes[3]
for label, mask, color in event_types:
    if np.sum(mask) > 10:
        data = vpd.values[mask]
        ax.hist(data, bins=20, alpha=0.5, label=label, color=color, edgecolor='black')

ax.set_xlabel('Afternoon VPD (kPa)')
ax.set_ylabel('Frequency')
ax.set_title('VPD Distribution by Event Type', fontweight='bold')
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle(f'Composite Analysis: {state_name} (Summer 2010-2023)',
            fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '04_composite_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
print("Plot 4 saved!")

## Plot 5: Principal Component Analysis

Reduce dimensionality and identify dominant patterns.

In [ ]:
state_name = 'Texas'
state_id = FOCUS_STATES[state_name]

# Collect all metrics
metrics_data = {
    'Day Heat (>30)': compute_state_mean(summer_day_heat, state_id, state_mask).values,
    'Day Heat (>35)': compute_state_mean(ds_day['hours_above_35'].where(ds_day.time.dt.month.isin([6,7,8]), drop=True), state_id, state_mask).values,
    'Night Poor Rec': compute_state_mean(summer_night_poor, state_id, state_mask).values,
    'VPD Mean': compute_state_mean(summer_vpd, state_id, state_mask).values,
    'VPD Max': compute_state_mean(ds_vpd['vpd_max'].where(ds_vpd.time.dt.month.isin([6,7,8]), drop=True), state_id, state_mask).values
}

# Create DataFrame and remove NaNs
df = pd.DataFrame(metrics_data)
df_clean = df.dropna()

# Standardize
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_clean)

# Perform PCA
pca = PCA()
pca_result = pca.fit_transform(df_scaled)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

# Plot 1: Explained variance
ax = axes[0]
ax.bar(range(1, len(pca.explained_variance_ratio_)+1), pca.explained_variance_ratio_*100,
      color='steelblue', alpha=0.7, edgecolor='black')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Explained Variance (%)')
ax.set_title('PCA Explained Variance', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

for i, var in enumerate(pca.explained_variance_ratio_*100):
    ax.text(i+1, var, f'{var:.1f}%', ha='center', va='bottom', fontsize=9)

# Plot 2: Cumulative variance
ax = axes[1]
cumvar = np.cumsum(pca.explained_variance_ratio_*100)
ax.plot(range(1, len(cumvar)+1), cumvar, marker='o', linewidth=2, markersize=8, color='darkgreen')
ax.axhline(90, color='red', linestyle='--', label='90% threshold')
ax.set_xlabel('Number of Components')
ax.set_ylabel('Cumulative Explained Variance (%)')
ax.set_title('Cumulative Variance Explained', fontweight='bold')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

# Plot 3: PC1 vs PC2 scatter
ax = axes[2]
scatter = ax.scatter(pca_result[:, 0], pca_result[:, 1], 
                    c=df_clean['Day Heat (>30)'], cmap='YlOrRd',
                    alpha=0.5, s=10)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('PCA Score Plot (PC1 vs PC2)', fontweight='bold')
ax.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax, label='Day Heat (>30°C)')

# Plot 4: Loading plot
ax = axes[3]
loadings = pca.components_[:2].T
feature_names = df_clean.columns

for i, (name, loading) in enumerate(zip(feature_names, loadings)):
    ax.arrow(0, 0, loading[0], loading[1], head_width=0.05, head_length=0.05,
            fc=plt.cm.tab10(i), ec=plt.cm.tab10(i), linewidth=2)
    ax.text(loading[0]*1.15, loading[1]*1.15, name, ha='center', va='center',
           fontsize=10, fontweight='bold')

ax.set_xlabel(f'PC1 Loading ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 Loading ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('PCA Loading Plot', fontweight='bold')
ax.axhline(0, color='black', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.5)
ax.grid(True, alpha=0.3)
ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)

plt.suptitle(f'Principal Component Analysis: {state_name} (Summer 2010-2023)',
            fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '05_pca_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
print("Plot 5 saved!")

## Plot 6: Bivariate Density Plots

Smooth density estimates of joint distributions.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

variable_pairs = [
    ('Day Heat (>30)', 'VPD Mean', 'YlOrRd'),
    ('Day Heat (>30)', 'Night Poor Rec', 'RdPu'),
    ('VPD Mean', 'Night Poor Rec', 'BuPu'),
    ('Day Heat (>35)', 'VPD Max', 'Reds')
]

for idx, (var1_name, var2_name, cmap) in enumerate(variable_pairs):
    ax = axes[idx]
    
    var1 = df_clean[var1_name].values
    var2 = df_clean[var2_name].values
    
    # Create KDE
    xy = np.vstack([var1, var2])
    kde = gaussian_kde(xy)
    
    # Create grid for contour plot
    x_min, x_max = var1.min(), var1.max()
    y_min, y_max = var2.min(), var2.max()
    xx, yy = np.mgrid[x_min:x_max:100j, y_min:y_max:100j]
    positions = np.vstack([xx.ravel(), yy.ravel()])
    density = np.reshape(kde(positions).T, xx.shape)
    
    # Plot contours
    contour = ax.contourf(xx, yy, density, levels=15, cmap=cmap, alpha=0.8)
    ax.contour(xx, yy, density, levels=10, colors='black', alpha=0.3, linewidths=0.5)
    
    ax.set_xlabel(var1_name)
    ax.set_ylabel(var2_name)
    ax.set_title(f'{var1_name} vs {var2_name}\nBivariate Density', fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    plt.colorbar(contour, ax=ax, label='Density')

plt.suptitle(f'Bivariate Density Plots: {state_name} (Summer 2010-2023)',
            fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '06_bivariate_density.png', dpi=300, bbox_inches='tight')
plt.show()
print("Plot 6 saved!")

## Plot 7: Conditional Distributions

How one variable's distribution changes conditioned on another.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

# Condition on VPD levels
vpd_data = df_clean['VPD Mean']
vpd_low = vpd_data < np.percentile(vpd_data, 33)
vpd_med = (vpd_data >= np.percentile(vpd_data, 33)) & (vpd_data < np.percentile(vpd_data, 67))
vpd_high = vpd_data >= np.percentile(vpd_data, 67)

conditions = [
    ('Low VPD (<P33)', vpd_low, 'blue'),
    ('Medium VPD (P33-P67)', vpd_med, 'orange'),
    ('High VPD (>P67)', vpd_high, 'red')
]

# Plot 1: Day heat distribution by VPD condition
ax = axes[0]
for label, mask, color in conditions:
    data = df_clean.loc[mask, 'Day Heat (>30)']
    ax.hist(data, bins=20, alpha=0.5, label=label, color=color, edgecolor='black', density=True)

ax.set_xlabel('Daytime Hours > 30°C')
ax.set_ylabel('Probability Density')
ax.set_title('Daytime Heat | VPD Level', fontweight='bold')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

# Plot 2: Night recovery by VPD condition  
ax = axes[1]
for label, mask, color in conditions:
    data = df_clean.loc[mask, 'Night Poor Rec']
    ax.hist(data, bins=20, alpha=0.5, label=label, color=color, edgecolor='black', density=True)

ax.set_xlabel('Nighttime Hours > 24°C')
ax.set_ylabel('Probability Density')
ax.set_title('Poor Recovery | VPD Level', fontweight='bold')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

# Condition on daytime heat
day_heat = df_clean['Day Heat (>30)']
heat_low = day_heat < np.percentile(day_heat, 33)
heat_med = (day_heat >= np.percentile(day_heat, 33)) & (day_heat < np.percentile(day_heat, 67))
heat_high = day_heat >= np.percentile(day_heat, 67)

heat_conditions = [
    ('Low Heat (<P33)', heat_low, 'green'),
    ('Medium Heat (P33-P67)', heat_med, 'yellow'),
    ('High Heat (>P67)', heat_high, 'red')
]

# Plot 3: VPD by heat condition
ax = axes[2]
for label, mask, color in heat_conditions:
    data = df_clean.loc[mask, 'VPD Mean']
    ax.hist(data, bins=20, alpha=0.5, label=label, color=color, edgecolor='black', density=True)

ax.set_xlabel('Afternoon VPD (kPa)')
ax.set_ylabel('Probability Density')
ax.set_title('VPD | Daytime Heat Level', fontweight='bold')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

# Plot 4: Night recovery by heat condition
ax = axes[3]
for label, mask, color in heat_conditions:
    data = df_clean.loc[mask, 'Night Poor Rec']
    ax.hist(data, bins=20, alpha=0.5, label=label, color=color, edgecolor='black', density=True)

ax.set_xlabel('Nighttime Hours > 24°C')
ax.set_ylabel('Probability Density')
ax.set_title('Poor Recovery | Daytime Heat Level', fontweight='bold')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

plt.suptitle(f'Conditional Distributions: {state_name} (Summer 2010-2023)',
            fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '07_conditional_distributions.png', dpi=300, bbox_inches='tight')
plt.show()
print("Plot 7 saved!")

## Plot 8: Multi-Metric Heat Stress Index

Develop a combined heat stress index using multiple variables.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

# Create composite index (normalized sum)
# Normalize each variable to 0-1 scale
def normalize(x):
    return (x - x.min()) / (x.max() - x.min())

df_norm = df_clean.copy()
for col in df_norm.columns:
    df_norm[col] = normalize(df_norm[col])

# Create index (weighted sum)
# Weights: emphasize daytime heat and VPD
weights = {
    'Day Heat (>30)': 0.3,
    'Day Heat (>35)': 0.25,
    'Night Poor Rec': 0.2,
    'VPD Mean': 0.15,
    'VPD Max': 0.1
}

heat_index = sum(df_norm[col] * weight for col, weight in weights.items())

# Plot 1: Index distribution
ax = axes[0]
ax.hist(heat_index, bins=50, color='darkred', alpha=0.7, edgecolor='black')
ax.axvline(heat_index.quantile(0.90), color='orange', linestyle='--', linewidth=2, label='P90')
ax.axvline(heat_index.quantile(0.95), color='red', linestyle='--', linewidth=2, label='P95')
ax.set_xlabel('Composite Heat Stress Index')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Heat Stress Index', fontweight='bold')
ax.legend(loc='best')
ax.grid(True, alpha=0.3, axis='y')

# Plot 2: Index time series (by year)
ax = axes[1]
years = pd.to_datetime(ds_day.time.values).year[df_clean.index]
annual_index = pd.DataFrame({'year': years, 'index': heat_index}).groupby('year')['index'].mean()

ax.plot(annual_index.index, annual_index.values, marker='o', linewidth=2, markersize=6, color='darkred')
z = np.polyfit(annual_index.index, annual_index.values, 1)
p = np.poly1d(z)
ax.plot(annual_index.index, p(annual_index.index), '--', linewidth=2, color='black',
       label=f'Trend: {z[0]:.4f}/yr')
ax.set_xlabel('Year')
ax.set_ylabel('Mean Heat Stress Index')
ax.set_title('Annual Average Heat Stress Index', fontweight='bold')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

# Plot 3: Index vs individual metrics
ax = axes[2]
for col in df_clean.columns[:3]:  # Plot first 3 metrics
    ax.scatter(df_clean[col], heat_index, alpha=0.3, s=5, label=col)

ax.set_xlabel('Original Metric Value')
ax.set_ylabel('Heat Stress Index')
ax.set_title('Index vs Component Metrics', fontweight='bold')
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)

# Plot 4: Categorize risk levels
ax = axes[3]
p33, p67, p90 = heat_index.quantile([0.33, 0.67, 0.90])

risk_categories = [
    ('Low', heat_index < p33, 'green'),
    ('Moderate', (heat_index >= p33) & (heat_index < p67), 'yellow'),
    ('High', (heat_index >= p67) & (heat_index < p90), 'orange'),
    ('Extreme', heat_index >= p90, 'red')
]

counts = [np.sum(mask) for _, mask, _ in risk_categories]
labels = [label for label, _, _ in risk_categories]
colors = [color for _, _, color in risk_categories]

bars = ax.bar(range(len(labels)), counts, color=colors, alpha=0.7, edgecolor='black')
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels)
ax.set_ylabel('Number of Days')
ax.set_title('Risk Category Distribution', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, counts):
    pct = val / len(heat_index) * 100
    ax.text(bar.get_x() + bar.get_width()/2., val,
           f'{val}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9)

plt.suptitle(f'Composite Heat Stress Index: {state_name} (Summer 2010-2023)',
            fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '08_heat_stress_index.png', dpi=300, bbox_inches='tight')
plt.show()
print("Plot 8 saved!")
print("\n=== Multivariate Relationships Analysis Complete ===")
print(f"8 plots saved to {OUTPUT_DIR.resolve()}")

## Summary

This notebook generated 8 comprehensive multivariate relationship plots:

1. **VPD-Temperature Scatter** - Strong positive correlation (r > 0.7)
2. **Day-Night Relationship** - Daytime heat predicts poor nighttime recovery
3. **Joint Probability** - Combined occurrence of multiple stressors
4. **Composite Analysis** - Compound extremes worse than single stressors
5. **PCA** - First 2 PCs explain >80% of variance
6. **Bivariate Density** - Smooth joint distributions reveal clustering
7. **Conditional Distributions** - VPD amplifies heat stress effects
8. **Heat Stress Index** - Composite metric shows increasing trend

**Key Insights:**
- VPD and temperature are strongly coupled (compound events)
- Compound extremes (high T + high VPD) occur 5-10% of summer days
- First PC represents overall heat stress intensity
- Heat stress index shows significant increasing trend
- Nighttime recovery strongly depends on daytime conditions